In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# 1. НАСТРОЙКИ
SUBMISSIONS_DIR = "submissions"
TARGET_MEAN = 278500.0

PATHS = {
    "y_30m":   f"{SUBMISSIONS_DIR}/081_30min_BEST_QUALITY_RAW.csv",
    "y_bq1h":  f"{SUBMISSIONS_DIR}/autogluon_best_quality_sub.csv",
    "y_spec":  f"{SUBMISSIONS_DIR}/autogluon_special_no_k.csv",
    "y_pulse": f"{SUBMISSIONS_DIR}/tf_global_pulse_sub.csv",
    "y_ridge": f"{SUBMISSIONS_DIR}/105_Ridge_Legion_RAW.csv"
}

# 2. ЗАГРУЗКА
print("Loading Exodia components for Strategy 3...")
dfs = {name: pd.read_csv(path).sort_values('id').reset_index(drop=True) for name, path in PATHS.items()}
base_ids = dfs["y_30m"]['id']
test_meta = pd.read_parquet('data/test_solo_track.parquet')[['id', 'route_id', 'timestamp']]
test_meta['timestamp'] = pd.to_datetime(test_meta['timestamp'])

# 3. ФУНКЦИЯ УМНОГО БЛЕНДИНГА (DISSENSUS)
print("Calculating Dynamic Dissensus Weights...")
all_preds = np.column_stack([dfs[n]['y_pred'].values for n in ["y_30m", "y_bq1h", "y_spec", "y_pulse", "y_ridge"]])

final_y = []
dissensus_counts = 0

for i in range(len(all_preds)):
    row = all_preds[i]
    mean_val = np.mean(row)
    std_val = np.std(row)
    cv = std_val / (mean_val + 1e-6) # Коэффициент вариации
    
    # Базовые веса (близкие к твоему лидеру 107)
    # Порядок: [y_30m, y_bq1h, y_spec, y_pulse, y_ridge]
    w = np.array([0.30, 0.20, 0.15, 0.20, 0.15])
    
    # ЕСЛИ МОДЕЛИ СИЛЬНО СПОРЯТ (CV > 0.3)
    if cv > 0.3:
        # Увеличиваем вес Ridge (самый стабильный) и снижаем вес Стэкинга
        w = np.array([0.20, 0.10, 0.10, 0.20, 0.40]) # Ridge получает 40%
        dissensus_counts += 1
    
    # Нормализуем веса
    w = w / w.sum()
    final_y.append(np.dot(row, w))

final_sub = pd.DataFrame({'id': base_ids, 'y_pred_raw': final_y})

# 4. КАЛИБРОВКА ОБЪЕМА
scaling_factor = TARGET_MEAN / final_sub['y_pred_raw'].mean()
final_sub['y_pred_calibrated'] = final_sub['y_pred_raw'] * scaling_factor

# 5. НАСЛЕДСТВО И ТИСКИ (Heritage 10:30)
train = pd.read_parquet('data/train_solo_track.parquet')
def get_heritage_30m(target_series):
    T = target_series.values.astype(np.float64)
    v30 = np.full(len(T), np.nan)
    zero_idx = np.where(T == 0)[0]
    if len(zero_idx) > 0:
        for idx in zero_idx:
            v30[idx] = 0
            if idx > 0: v30[idx-1] = 0
        for _ in range(3):
            for i in range(1, len(v30)):
                if np.isnan(v30[i]) and not np.isnan(v30[i-1]): v30[i] = max(0, T[i] - v30[i-1])
            for i in range(len(v30)-1, 0, -1):
                if np.isnan(v30[i-1]) and not np.isnan(v30[i]): v30[i-1] = max(0, T[i] - v30[i])
    return np.nan_to_num(v30, nan=T[-1]/2)[-1]

heritage_dict = {rid: get_heritage_30m(train[train['route_id'] == rid].sort_values('timestamp')['target_1h']) for rid in range(1000)}

final_sub = final_sub.merge(test_meta, on='id')
final_sub['v1030'] = final_sub['route_id'].map(heritage_dict)

# Тиски 11:00
mask_1100 = (final_sub['timestamp'].dt.hour == 11) & (final_sub['timestamp'].dt.minute == 0)
final_sub['y_pred'] = final_sub['y_pred_calibrated']
final_sub.loc[mask_1100, 'y_pred'] = np.maximum(final_sub.loc[mask_1100, 'y_pred'], final_sub.loc[mask_1100, 'v1030'])

# 6. ХАК: МАТЕМАТИЧЕСКАЯ ТЮРЬМА (Triple Window Consistency)
print("Applying Triple Window Mathematical Sanity Check...")
fix_count = 0
for rid in tqdm(range(1000)):
    # Выделяем 8 шагов этого маршрута
    route_idx = final_sub[final_sub['route_id'] == rid].index
    preds = final_sub.loc[route_idx, 'y_pred'].values
    
    # T[t] + T[t+2] >= T[t+1]
    for t in range(0, 6): # Проверяем все тройки внутри 8 шагов
        if preds[t] + preds[t+2] < preds[t+1]:
            # Корректируем провал в середине (делаем среднее между краями)
            # Это физически корректнее, чем оставлять "горб"
            preds[t+1] = preds[t] + preds[t+2]
            fix_count += 1
    
    final_sub.loc[route_idx, 'y_pred'] = preds

# 7. СОХРАНЕНИЕ
out_path = f"{SUBMISSIONS_DIR}/108_STRATEGY3_DISSENSUS_FIXED.csv"
final_sub[['id', 'y_pred']].sort_values('id').to_csv(out_path, index=False)

print(f"\n--- STRATEGY 3 REPORT ---")
print(f"Rows with high Dissensus: {dissensus_counts} ({dissensus_counts/8000:.1%})")
print(f"Triple Window violations fixed: {fix_count}")
print(f"Final Mean Volume: {final_sub['y_pred'].mean():.2f}")
print(f"🚀 MISSION READY: {out_path}")

Loading Exodia components for Strategy 3...
Calculating Dynamic Dissensus Weights...
Applying Triple Window Mathematical Sanity Check...


100%|██████████| 1000/1000 [00:00<00:00, 2137.82it/s]


--- STRATEGY 3 REPORT ---
Rows with high Dissensus: 235 (2.9%)
Triple Window violations fixed: 0
Final Mean Volume: 278502.84
🚀 MISSION READY: submissions/108_STRATEGY3_DISSENSUS_FIXED.csv
